# "Gaze-angle dependency of pupil-size measurements in head-mounted eye tracking"
## Bernhard Petersch & Kai Dierkes, Pupil Labs
This notebook contains Python code implementing all steps of the pupillometry analysis proposed in above mentioned manuscript.<br>
Install required packages via 'pip install -r requirements.txt'.<br>
The code cells are structured in 3 sections<br>

A - Import of required packages, dependencies<br>
B - Definition of data loading, processing and plotting routines<br>
C - Execution of entire data processing pipeline<br>

which, when executed sequentially from top to bottom will reproduce the main real world data result as shown in Fig.8 (panels A and C) of the manuscript.<br>
For ease of cross-reference, code cells in section C refer to the sections of the manuscript which detail the corresponding data processing steps.

###  A - Imports & Dependencies

In [1]:
import cv2
import enum
import glob
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import msgpack
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from pupil_detectors import Detector2D
import pyarrow
import pye3d.eye_model
from pye3d.refraction import Refractionizer
from pye3d.camera import CameraModel
from pye3d.observation import Observation
from pye3d.geometry.primitives import Ellipse
import scipy.stats
import sys
import typing as T

### B - Data loading routines

In [2]:
class Environment(enum.Enum):
    TIME = (60, 120)  # 15-45 interval of varying gaze-angles


class Trial(T.NamedTuple):
    video_path: str
    eye_id: int
    environment: Environment


def add_trials(run):
    """
    Collect trials (one per eye) from a recording run.
    """
    trials = []
    env = Environment.TIME

    for dir_ in run.iterdir():
        if (dir_ / 'eye0_timestamps.npy').is_file():
            trials.append(
                Trial(
                    video_path=dir_,
                    eye_id=0,
                    environment = env
                    
                )
            )

        if (dir_ / 'eye1_timestamps.npy').is_file():
            trials.append(
                Trial(
                    video_path=dir_,
                    eye_id=1,
                    environment = env
                )
            )

    return trials


def get_all_trials(root_data_directory):
    """
    Collect all trials across runs.
    """
    all_trials = []

    for run in sorted([x for x in Path(root_data_directory).iterdir() if x.is_dir()]):
        all_trials += add_trials(run)

    return all_trials

### C - Pupil contour extraction routines

In [3]:
def shift_pupil_detections(pupil_detections, eye_id, video_path, resolution=(400,400)):
    
    """
    Load measured eye camera intrinsics from file and correct pupil datums
    for eye camera image sensor center shift.

    Args:
        pupil_detections (list of dictionaries): previously calculated pupil datums
        eye_id (int): side of eye (0 or 1)
        video_path (pathlib Path): data directory of trial
        resolution (int tuple): image resolution [pixels] (Default=(400,400))
    """
    
    # load camera matrix for correcting center shift     
    with (video_path / "camera_properties.p").open("rb") as f:
            camera_properties = msgpack.unpack(f, raw=False)
    
    if eye_id == 0:
        camera_matrix = np.array(camera_properties['right']['camera_matrix'])
    elif eye_id == 1:
        camera_matrix = np.array(camera_properties['left']['camera_matrix'])
    
    width, height = resolution
    center_shift = camera_matrix[:2, 2]
    dx = width / 2. - center_shift[0]
    dy = height / 2. - center_shift[1]
           
    for i in range(len(pupil_detections)):
        pupil_detections[i] = apply_shift(pupil_detections[i], dx, dy)

    return pupil_detections

def apply_shift(pd, dx, dy):
    
    x, y = pd["ellipse"]["center"]
    pd["ellipse"]["center"] = (x + dx, y + dy)
    
    return pd

def add_timestamps(pupil_detections, eye_id, video_path):
    
    """
    Adds timestamp information to pupil datums.
    Timestamp files contain exactly one timestamp per video frame and thus per pupil detection.

    Args:
        pupil_detections (list of dictionaries): previously calculated pupil datums
        eye_id (int): side of eye (0 or 1)
        video_path (pathlib Path): data directory of trial
    """
    
    timestamps = np.load(video_path / f"eye{eye_id}_timestamps.npy")
    timestamps -= timestamps[0]      # subtract first timestamp from all timestamps
     
    for ts, detection in zip(timestamps, pupil_detections):
        detection["timestamp"] = ts
        
    return pupil_detections

def extract_pupil_contours(
    trial, center_shift=True, output_directory="analysis"
):

    """
    Loads eye video, segments pupil from every image and saves results.

    Args:
        trial (class Trial): object containing data directory of trial (video_path, pathlib Path),
        eye_id (side of eye, int 0 or 1) and start and end times of sweep period of experiment
        (class EnvironmentSweepPeriod, numeric values in seconds)
        center_shift (bool): whether to correct image sensor center shift (Default=True)
        output_directory (string): subdirectory to create for saving results
    """
        
    video_path = trial.video_path
    eye_id = trial.eye_id
    print(str(video_path), eye_id)
    
    # INITIALIZE pupil detector
    detector = Detector2D()
    detector.update_properties(
        {
            "2d": {
                "intensity_range": 23,
                "pupil_size_max": 130,
                "pupil_size_min": 20
            }
        }
    )

    # LOAD eye video file
    file_ = video_path / f"eye{eye_id}.mp4"
    cap = cv2.VideoCapture(str(file_))
    ret, frame = cap.read()
    
    if not cap.isOpened():
        raise FileNotFoundError(f"Video not found: {file_}")
        
    # EXTRACT pupil contours
    pupil_detections = []

    while ret:

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        pd_ = detector.detect(gray)
        pupil_detections.append(pd_)

        ret, frame = cap.read()

    # add TIMESTAMPS to pupil datums 
    pupil_detections = add_timestamps(pupil_detections, eye_id, video_path)
        
    # COMPENSATE camera sensor center shift
    if center_shift:        
        pupil_detections = shift_pupil_detections(pupil_detections, eye_id, video_path)
        
    # SAVE pupil detections to file
    if not (video_path / output_directory).is_dir:
        (video_path / output_directory).mkdir()
       
    with (video_path / output_directory / f"eye{eye_id}_2d_detections.p").open("wb") as f:
        msgpack.pack(pupil_detections, f, use_bin_type=True)
    

### D - 3D eye model initialization

In [4]:
def filter_by_time(pupil_detections, start_time, end_time):
    
    """
    Selects subset of pupil datums based on timestamp range.

    Args:
        pupil_detections (list of dictionaries): list of previously calculated pupil datums
        start_time (numeric): lower bound timestamp [s]
        end_time (numeric): upper bound timestamp [s]
    """
    
    return [
        pd
        for pd in pupil_detections
        if (start_time < pd["timestamp"] < end_time)
    ]

def get_best_pupil_detections(pupil_detections, percentage=0.1):

    """
    Selects subset of pupil datums with highest detection/segmentation confidence.

    Args:
        pupil_detections (list of dictionaries): previously calculated pupil datums
        percentage (float): fraction of pupil datums of highest confidence to return
    """
    
    confidences = np.asarray([pd["confidence"] for pd in pupil_detections])
    N = int(percentage * len(pupil_detections))
    best_idx = np.argsort(confidences)[-N:]
    best_pupil_detections = [pupil_detections[i] for i in best_idx]
    return best_pupil_detections

def convert_pupil_datum_observation(pupil_datum, resolution=(400,400), focal_length=575.):
    
    """
    Converts pupil datum data as provided by pupil_detectors.Detector2D to Observation
    data object for processing by pye3d.eye_model.
    Ellipse angle is converted from the definition used in OpenCV to the one used in pye3d.

    Args:
        pupil_datum (dictionary): pupil datum
        resolution (int tuple): eye video image resolution [pixels] (Default=(400,400))
        focal_length (float): eye camera focal length [pixels] (Default=575.)
    """
    
    width, height = resolution
    # converting from image centered to camera centered [0-400],[0-400] -> [-200,200],[-200,200]
    center = (
        pupil_datum["ellipse"]["center"][0] - width / 2,
        pupil_datum["ellipse"]["center"][1] - height / 2,
    )
    minor_radius = pupil_datum["ellipse"]["axes"][0] / 2.0
    major_radius = pupil_datum["ellipse"]["axes"][1] / 2.0
    angle = (pupil_datum["ellipse"]["angle"] - 90.0) * np.pi / 180.0
    ellipse = Ellipse(center, minor_radius, major_radius, angle)

    return Observation(
        ellipse,
        pupil_datum["confidence"],
        pupil_datum["timestamp"],
        focal_length,
    )

def nandivision(n, d):
    return n / d if d else np.nan

def build_eyemodel(trial,output_directory="analysis"):
    """ 
    Loads pupil detection, calculates eyeball center position based on given timestamp range
    and saves results for further refraction corrected pupil radii calculations
    
    Args:
        trial (class Trial): object containing data directory of trial (video_path, pathlib Path),
        eye_id (side of eye, int 0 or 1) and start and end times of timestamp range from which to
        calculate eyeball center (class EnvironmentSweepPeriod, numeric values in seconds)
        output_directory (string): subdirectory to use for saving results
    """
    video_path = trial.video_path
    eye_id = trial.eye_id
    start_time,end_time = trial.environment.value
    print(str(video_path), eye_id, start_time, end_time)
    
    analysis_path = video_path / output_directory
    analysis_path.mkdir(exist_ok = True)
    
    with (analysis_path / f"eye{eye_id}_2d_detections.p").open("rb") as f:
        pupil_detections = msgpack.unpack(f, raw=False)

    # CALCULATE eyeball position based on pre-defined time interval ("sweep")
    # using only 10% pupil detections with highest confidence

    pupil_detections_sweep = filter_by_time(pupil_detections, start_time, end_time)
    best_pupil_detections = get_best_pupil_detections(pupil_detections_sweep, 0.1)

    camera = CameraModel(focal_length=575., resolution=(400, 400))
    eyemodel = pye3d.eye_model.TwoSphereModel(camera=camera)

    for pupil_datum in best_pupil_detections:
        observation = convert_pupil_datum_observation(pupil_datum,camera.resolution, camera.focal_length)
        eyemodel.add_observation(observation)

    sphere_center = eyemodel.estimate_sphere_center().three_dim
    
    data = {
        "eyemodel": eyemodel,
        "sphere_center": sphere_center
    }
    
    with (analysis_path / f"eye{eye_id}_eyemodel.pkl").open("wb") as f:
        pickle.dump(data, f)

    
def extract_and_save_predictions(trial,eyemodel_data=None, output_directory="analysis"):
    video_path= trial.video_path
    eye_id = trial.eye_id
    
    if eyemodel_data is None:
        with(video_path / output_directory / f"eye{eye_id}_eyemodel.pkl").open("rb") as file:
            eyemodel_data = pickle.load(file)
            
    eyemodel = eyemodel_data["eyemodel"]
    sphere_center = eyemodel_data["sphere_center"]
    
    
    with(video_path / output_directory / f"eye{eye_id}_2d_detections.p").open("rb") as file:
        pupil_detections = msgpack.unpack(file,raw=False)
        
    refractionizer = Refractionizer()
    
    # CALCULATE pupil radii and gaze vectors

    circles_3d = np.asarray([
            eyemodel.predict_pupil_circle(
                convert_pupil_datum_observation(pupil_datum)
            )
            for pupil_datum in pupil_detections
    ])
    
    gaze_vectors = np.asarray([circle.normal for circle in circles_3d])
    radii = np.asarray([circle.radius for circle in circles_3d])
    radii.shape = -1, 1

    
    # CALCULATE refraction corrected pupil radii

    N = len(circles_3d)
    sphere_center_column = np.repeat([sphere_center], repeats=N, axis=0)
    
    X = np.hstack((sphere_center_column, gaze_vectors, radii))
    radii_refraction = refractionizer.correct_radius(X)[:, 0]

    # SAVE data

    df = pd.DataFrame(
        [
            {
                "circularity": nandivision(
                    entry["ellipse"]["axes"][0], entry["ellipse"]["axes"][1]
                ),
                "confidence": entry["confidence"],
                "timestamp": entry["timestamp"],
            }
            for entry in pupil_detections
        ]
    )
    df["radius_refraction"] = radii_refraction
    df.to_parquet(
        (
            video_path / output_directory / f"eye{eye_id}_2d_predictions.p"
        )
    )
    

### E - Pupil size baselines calculation routine

In [5]:
def calculate_pupil_size_baselines(all_trials, conf_thresh=0.6, output_directory="analysis", SAVE=True):

    """
    Loads pupil size calculations, calculates baselines from timestamp range stored
    in metadata of each trial and saves data into a single dataframe.

    Args:
        all_trials: list of data directories and metadata of individual
        recordings as produced using get_all_trials(root_data_directory)
        conf_thresh (float): confidence threshold for BASELINE pupil size calculation (Default=0.6)
        output_directory (string): subdirectory to load pupil size data from
    """

    frames = []

    for trial in all_trials:

        try:

            folder = trial.video_path
            eye_id = trial.eye_id
            start_time, end_time = trial.environment.value
            
            # extract test subject ID
            # parts[-2] = 'subject_05-B1'
            name = folder.parts[-2].split('-')[0]

            # extract time range for baseline calculation
            t_start_sweep = start_time
            t_sweep_duration = end_time - start_time

            # LOAD pupil size data

            dataframe = pd.read_parquet(folder / output_directory / f"eye{eye_id}_2d_predictions.p")

            condition = (dataframe.confidence > conf_thresh) & (~pd.isna(dataframe.circularity))
            
            # CALCULATE pupil size baselines from sweep period and add to dataframe

            sweep_period = (dataframe.timestamp.between(t_start_sweep, t_start_sweep+t_sweep_duration)) & condition

            metrics = ["radius", "radius_refraction", "major"]
            for met in metrics:
                dataframe[f"baseline_{met}"] = dataframe.loc[sweep_period, met].median(skipna=True)

            dataframe['eye'] = "left" if eye_id == 1 else "right"
            dataframe['name'] = name
            dataframe['t_end_baseline'] = t_start_sweep
            
            dataframe = dataframe[(~pd.isna(dataframe.circularity))]
            frames.append(dataframe)

        except Exception as e:
            print("Exception:", e)
            print("... was thrown in trial:", str(video_path), "!")

    dataframe_all = pd.concat(frames)
    
    # SAVE data
    if SAVE:
        dataframe_all.to_parquet("dataframe_all_code_example.p")
    
    return dataframe_all

## Execute data processing pipeline

Set data root path and number of CPUs to use (single-thread or multi-core parallel processing)

In [6]:
root_path = "C:/Users/stock/Documents/Master BME/Masterarbeit/6_Gaze_angle_model/data"
#root_path = "C:/Users/stock/Documents/GitHub/petersch_dierkes_brm_2021_data_materials/data"
n_jobs = 1
#n_jobs = 8

### Get recording data folders

In [7]:
all_trials = get_all_trials(root_path)
print(f"Number of individual recordings: {len(all_trials)}")

Number of individual recordings: 2


### Extract pupil contours

In [8]:
if n_jobs == 1:
    for trial in all_trials[:2]:
        extract_pupil_contours(trial)
else:
    result = Parallel(n_jobs=n_jobs)(delayed(extract_pupil_contours)(trial) for trial in all_trials[:2])

C:\Users\stock\Documents\Master BME\Masterarbeit\6_Gaze_angle_model\data\subject_01\001 0
C:\Users\stock\Documents\Master BME\Masterarbeit\6_Gaze_angle_model\data\subject_01\001 1


### Calculate pupil sizes

In [16]:
if n_jobs == 1:
    for trial in all_trials[:2]:
        #extract_predictions(trial)
        build_eyemodel(trial)
        extract_and_save_predictions(trial)
else:
    result = Parallel(n_jobs=n_jobs)(delayed(extract_predictions)(trial) for trial in all_trials[:2])

C:\Users\stock\Documents\Master BME\Masterarbeit\6_Gaze_angle_model\data\subject_01\001 0 60 120
C:\Users\stock\Documents\Master BME\Masterarbeit\6_Gaze_angle_model\data\subject_01\001 1 60 120


In [ ]:
dataframe_all = calculate_pupil_size_baselines(all_trials)

## Hayes and Petrov

## Comparing Petersch, Hayes and Tobii

### Ideas

Remove data with detection confidence < 97%  
Add velocity filter <br>
Rebuild missing data using interpolation ~ Zandi